NVIDIA

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import PromptTemplate
from function import sendMail
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatNVIDIA(
    api_key = os.getenv("nvidia_key"),
    model = "meta/muse-glimmer-30b",
)

llm = llm.bind_tools([sendMail])

while True:
    user_input = input("Ask: ")
    if user_input.lower() == 'quit':
        print("Agent: GoodBye!")
        exit(0)
    
    prompt = PromptTemplate(
        template = '''
        Please modify the body of the email intelligently.

        Rules:
        - Make the message professional and polite.
        - Correct grammar and spelling.
        - Keep the message short and meaningful.
        - Do not change the original intention.
        - Then send the email using the sendMail tool.
        user's request: {user_input}''',
        input_variables = ["user_input"]
    )
    
    myMessage = prompt.invoke({"user_input": user_input})
    responses = llm.invoke(myMessage)
    
    if responses.tool_calls:
        tool_name = responses.tool_calls[0]['name']
        tool_args = responses.tool_calls[0]['args']
        
        if tool_name == "sendMail":
            result = sendMail.invoke(tool_args)
            print("Agent:", result)
        else:
            print("I can't help you except sending mail.")

# read time out

GEMINI

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate 
from function import sendMail
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    api_key = os.getenv("gemini_key"),
    model = "gemini-3.6-flash"
)

llm = llm.bind_tools([sendMail])

while True:
    user_input = input("Agent: ")
    if user_input.lower() == 'exit':
        print("Agent: See ya!")
        exit(0)
    instructions = PromptTemplate(
        template = '''
        Please modify the body of the email intelligently.

        Rules:
        - Make the message professional and polite.
        - Correct grammar and spelling.
        - Keep the message short and meaningful.
        - Do not change the original intention.
        - Then send the email using the sendMail tool.
        user's request: {user_input}''',
        input_variables = ["user_input"]
     )
    
    myMessage = instructions.invoke({"user_input": user_input})
    responses = llm.invoke(myMessage)
    
    if responses.tool_calls:
        tool_name = responses.tool_calls[0]['name']
        tool_args = responses.tool_calls[0]['args']
        
        if tool_name == "sendMail":
            answer = sendMail.invoke(tool_args)
            print("Agent:", answer)
        else:
            print("I can't help with that.")

OPENAI

In [ ]:
import streamlit as st
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from function import sendMail
from langchain_core.prompts import PromptTemplate

load_dotenv()

# Connecting to LLM
llm = ChatOpenAI(
    api_key=os.getenv("openai_key"),
    model="gpt-4.1-mini",
    temperature=1
)
llm = llm.bind_tools([sendMail])
# Streamlit UI
st.title("Smart Mail Agent")

user_input = st.text_area("Enter your message:")

if st.button("Submit"):
    if user_input.strip():
        # Create Prompt
        prompt = PromptTemplate(
            template="""
                Please modify the body of the email intelligently.
                Rules:
                - Make the message professional and polite.
                - Correct grammar and spelling.
                - Keep the message short and meaningful.
                - Do not change the original intention.
                - Then send the email using the sendMail tool.
                User's request:{user_input}
            """,
            input_variables=["user_input"]
        )
        myMessages = prompt.invoke({
            "user_input": user_input
        })
        # Connected to LLM
        response = llm.invoke(myMessages)

        if response.tool_calls:
            tool_name = response.tool_calls[0]["name"]
            tool_args = response.tool_calls[0]["args"]

            if tool_name == "sendMail":
                result = sendMail.invoke(tool_args)

                st.success("Email sent successfully!")
                st.write(result)
        else:
            st.warning(
                "Sorry, I can only help with sending smart messages through mail."
            )
    else:
        st.warning("Please enter a message.")
